In [1]:
%%writefile ../src/evaluate.py
"""Metrics and cross-validation helpers. Everything here works on TRAIN data only."""
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from src import config


def make_cv(n_splits: int = 5):
    """Same grouped + stratified logic as the train/test split."""
    return StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                random_state=config.RANDOM_STATE)


def oof_predictions(model, X, y, groups, n_splits: int = 5) -> np.ndarray:
    """Out-of-fold P(late): each row is predicted by a model that never saw it
    (or any order of the same customer)."""
    oof = np.zeros(len(y))
    for tr, va in make_cv(n_splits).split(X, y, groups):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return oof


def score_row(y, p, threshold: float = 0.5) -> dict:
    """Standard scorecard for one set of predicted probabilities."""
    pred = (p >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
        "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        "brier": brier_score_loss(y, p),
        "accuracy@0.5": accuracy_score(y, pred),
        "precision@0.5": precision_score(y, pred, zero_division=0),
        "recall@0.5": recall_score(y, pred, zero_division=0),
    }


def fold_scores(model, X, y, groups, n_splits: int = 5) -> pd.DataFrame:
    """Score per fold, to see how stable a result is."""
    rows = []
    for i, (tr, va) in enumerate(make_cv(n_splits).split(X, y, groups), 1):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        rows.append({"fold": i, **score_row(y.iloc[va], p)})
    return pd.DataFrame(rows).set_index("fold")

Overwriting ../src/evaluate.py


In [2]:
import sys
sys.path.append("..")
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

from src import config
from src.evaluate import fold_scores

train = pd.read_parquet(config.ROOT / "data/processed/train.parquet")
X = train[[config.SHIP_MODE]]
y = train[config.TARGET]
groups = train["Customer Id"]
print(X.shape, y.mean().round(4))

(138213, 1) 0.5729


In [3]:
majority = DummyClassifier(strategy="prior")

mode_only = Pipeline([
    ("prep", ColumnTransformer([("onehot", OneHotEncoder(handle_unknown="ignore"),
                                 [config.SHIP_MODE])])),
    ("clf", LogisticRegression(max_iter=1000)),
])

results = {}
for name, model in [("majority", majority), ("mode only", mode_only)]:
    f = fold_scores(model, X, y, groups)
    results[name + " (mean)"] = f.mean()
    results[name + " (std)"] = f.std()

summary = pd.DataFrame(results).round(4)
summary

,majority (mean),majority (std),mode only (mean),mode only (std)
roc_auc,0.5000,0.0,0.7411,0.0049
pr_auc,0.5729,0.0,0.7840,0.0029
log_loss,0.6825,0.0,0.5380,0.0033
brier,0.2447,0.0,0.1884,0.0014
accuracy@0.5,0.5729,0.0,0.6963,0.0036
precision@0.5,0.5729,0.0,0.8848,0.0053
recall@0.5,1.0000,0.0,0.5404,0.0078


In [4]:
mode_only.fit(X, y)
modes = pd.DataFrame({config.SHIP_MODE: ["First Class", "Second Class", "Same Day", "Standard Class"]})
learned = pd.Series(mode_only.predict_proba(modes)[:, 1],
                    index=modes[config.SHIP_MODE], name="predicted P(late)")
learned.round(4)

Shipping Mode
First Class       0.9997
Second Class      0.7947
Same Day          0.4816
Standard Class    0.3982
Name: predicted P(late), dtype: float64